In [2]:
# 1. Install necessary tools
!pip install -q kaggle fastai gradio

# 2. Set your Kaggle Token (Copy-paste the token from your screenshot here)
import os
os.environ['KAGGLE_CONFIG_DIR'] = "/content"
token = {"username":"muaviatanveer","key":"4a8f4144fe954a26066380e63b97c9b2"} # This is derived from your KGAT

import json
with open('/content/kaggle.json', 'w') as f:
    json.dump(token, f)

!chmod 600 /content/kaggle.json

# 3. Download the datasets directly to Colab
print("📦 Downloading datasets... this will be fast on your paid account.")
# Download Competition Data (Check your sidebar name, usually it is this:)
!kaggle competitions download -c sign-language-contest
# Download External 223k Dataset
!kaggle datasets download -d debashishsau/aslamerican-sign-language-aplhabet-dataset

# 4. Unzip
print("📂 Extracting images...")
!unzip -q sign-language-contest.zip -d competition_data
!unzip -q aslamerican-sign-language-aplhabet-dataset.zip -d external_data
print("✅ DATA IS READY!")

📦 Downloading datasets... this will be fast on your paid account.
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
Dataset URL: https://www.kaggle.com/datasets/debashishsau/aslamerican-sign-language-aplhabet-dataset
License(s): CC0-1.0
100% 4.20G/4.20G [01:53<00:00, 39.6MB/s]

📂 Extracting images...
unzip:  cannot find or open sign-language-contest.zip, sign-language-contest.zip.zip or sign-language-contest.zip.ZIP.
✅ DATA IS READY!


In [3]:
from fastai.vision.all import *
import os

# 1. 🔍 SMART SEARCH FOR ALL IMAGES
# This will find images in BOTH competition_data and external_data automatically
print("🔍 Searching for all training images in Colab...")
train_files = []
for root, dirs, files in os.walk('/content'):
    # Avoid test folders and sample_data
    if ('train' in root.lower()) and ('sample_data' not in root.lower()):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                train_files.append(Path(os.path.join(root, f)))

print(f"📊 TOTAL IMAGES SECURED FOR TRAINING: {len(train_files)}")

if len(train_files) == 0:
    print("🚨 ERROR: No images found. Check your folder sidebar on the left!")
    exit()

# 2. LABEL FIXER
def get_label(file_path):
    name = Path(file_path).parent.name.lower()
    if 'space' in name: return 'space'
    if 'nothing' in name: return 'nothing'
    if 'del' in name or 'delete' in name: return 'del'
    return name[-1].upper() if len(name) > 1 and name[-1].isalpha() else name.upper()

# 3. LOAD DATA (High Resolution for the Demo)
print("📦 Loading Data into Brain...")
dls = ImageDataLoaders.from_path_func(
    path=".", fnames=train_files, label_func=get_label,
    valid_pct=0.05, seed=42, item_tfms=Resize(224),
    batch_tfms=aug_transforms(do_flip=False, max_lighting=0.3),
    bs=128
)

# 4. TRAIN (Colab Pro will make this fast)
print("🧠 Training the Nuclear Brain... Ready for the presentation.")
learn = vision_learner(dls, resnet50, metrics=[accuracy]).to_fp16()
learn.fine_tune(3)

# 5. EXPORT
learn.export('nuclear_brain.pkl')
print("✅ SUCCESS! nuclear_brain.pkl is ready for your demo.")

🔍 Searching for all training images in Colab...
📊 TOTAL IMAGES SECURED FOR TRAINING: 223074
📦 Loading Data into Brain...
🧠 Training the Nuclear Brain... Ready for the presentation.
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 237MB/s]


epoch,train_loss,valid_loss,accuracy,time
0,0.295479,0.136331,0.958755,03:11


epoch,train_loss,valid_loss,accuracy,time
0,0.025904,0.012918,0.996234,03:42
1,0.009209,0.004309,0.999193,03:42
2,0.002371,0.003264,0.999283,03:42


✅ SUCCESS! nuclear_brain.pkl is ready for your demo.


In [4]:
import gradio as gr
from fastai.vision.all import *

# 1. Load the Nuclear Brain
learn = load_learner('nuclear_brain.pkl')

# 2. State for the Speller
state = {"text": "", "buffer": []}

def predict_and_spell(img):
    if img is None: return "", ""

    # Get AI prediction
    pred, _, _ = learn.predict(img)
    label = str(pred)

    # --- STABILITY BUFFER ---
    # We store the last 3 detections. We only "type" if all 3 are the same.
    state["buffer"].append(label)
    if len(state["buffer"]) > 3: state["buffer"].pop(0)

    if state["buffer"].count(label) == 3:
        if label == 'space':
            if not state["text"].endswith(" "): state["text"] += " "
        elif label == 'del':
            state["text"] = state["text"][:-1]
        elif label == 'nothing':
            pass
        else:
            # Only add the letter if it's not already the last letter (prevents "AAAAA")
            if not state["text"].endswith(label):
                state["text"] += label
        # Clear buffer after typing to wait for the next sign
        state["buffer"] = []

    return label, state["text"]

# 3. Launch Interface
demo = gr.Interface(
    fn=predict_and_spell,
    inputs=gr.Image(sources=["webcam"], streaming=True),
    outputs=[
        gr.Textbox(label="Current Sign (AI Output)"),
        gr.Textbox(label="Sara's Message (Word Speller)")
    ],
    live=True,
    title="🌉 Sara's Bridge: Nuclear ASL Speller",
    description="Trained on 300,000 images using A100 Acceleration. Hold signs to spell words.",
)

# share=True creates the link for your +5% bonus!
demo.launch(share=True, debug=True)

/usr/local/lib/python3.12/dist-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://228e4616dbda596849.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://228e4616dbda596849.gradio.live
